# 06 — Multi-horizon deep-learning models

This notebook trains and evaluates the deep-learning family used to forecast greenhouse air temperature and relative humidity at five temporal resolutions and four forecast horizons.

Four architectures are compared: **MLP, LSTM, GRU, and TCN**. Every architecture receives the same 24-hour `SHT_TIME` sequence and predicts the eight target–horizon combinations simultaneously. Two predefined hyperparameter candidates are evaluated per architecture with three independent seeds (`2026`, `2027`, and `2028`).

Model selection uses validation data only. The selected candidate for each architecture and temporal resolution is refitted on `train + validation`, and its test prediction is the mean of three independently fitted seed models. The final deep-learning representative is selected separately for each forecasting task using validation performance and seed stability.

Run notebooks `01`–`05` before this notebook. Training is computationally intensive and uses deterministic CPU execution to reduce hardware-dependent variation.

**Inputs**

- Resolution datasets in `data/processed/resolutions/`
- Effective forecast-origin indices in `data/processed/effective_indices/`

**Main outputs**

- Predictions, metrics, and training histories in `results/deep_learning/`
- Trained neural networks in `models/deep_learning/`
- Fitted preprocessing objects in `preprocessors/deep_learning/`
- Summary and training-curve figures in PNG and PDF format in `figures/deep_learning/`

All paths are relative to the repository root.


## Dependency note

Use the project environment specified in `requirements.txt`. If an import fails, activate the `greenhouse-manuscript` environment, restart the kernel, and run the notebook again from the beginning. Packages are not installed from inside the notebook.


In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import random
import time
import warnings

BASE_SEED = 2026

# Configure deterministic CPU execution before importing TensorFlow.
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_CUDNN_DETERMINISTIC"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_NUM_INTRAOP_THREADS"] = "1"
os.environ["TF_NUM_INTEROP_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ.setdefault("PYTHONHASHSEED", str(BASE_SEED))

for package in ["tensorflow", "tqdm"]:
    if importlib.util.find_spec(package) is None:
        raise ImportError(
            f"{package} is not installed in the active kernel. "
            "Install the project requirements and restart the kernel."
        )

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from tensorflow.keras import layers

# Use Jupyter widgets when available and fall back silently to a text progress bar.
try:
    from tqdm.notebook import IProgress, tqdm as notebook_tqdm

    if IProgress is None:
        raise ImportError
    tqdm = notebook_tqdm
except (ImportError, AttributeError):
    from tqdm import tqdm

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid", context="notebook")

try:
    tf.config.set_visible_devices([], "GPU")
except RuntimeError as error:
    raise RuntimeError(
        "TensorFlow was initialized before CPU configuration. Restart the kernel."
    ) from error

tf.config.threading.set_intra_op_parallelism_threads(1)
tf.config.threading.set_inter_op_parallelism_threads(1)
tf.config.experimental.enable_op_determinism()
random.seed(BASE_SEED)
np.random.seed(BASE_SEED)
tf.keras.utils.set_random_seed(BASE_SEED)

print("TensorFlow configured for deterministic CPU execution.")


In [ ]:
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "processed" / "resolutions").exists():
            return candidate
    raise FileNotFoundError(
        "Repository root not found. Run notebook 03 first and keep the standard folder structure."
    )


PROJECT_ROOT = find_project_root()
RESOLUTION_DIR = PROJECT_ROOT / "data" / "processed" / "resolutions"
INDEX_DIR = PROJECT_ROOT / "data" / "processed" / "effective_indices"
RESULTS_DIR = PROJECT_ROOT / "results" / "deep_learning"
MODEL_DIR = PROJECT_ROOT / "models" / "deep_learning"
PREPROCESSOR_DIR = PROJECT_ROOT / "preprocessors" / "deep_learning"
FIGURE_DIR = PROJECT_ROOT / "figures" / "deep_learning"

for directory in [RESULTS_DIR, MODEL_DIR, PREPROCESSOR_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Resolution datasets: {RESOLUTION_DIR.relative_to(PROJECT_ROOT)}")
print(f"Effective indices: {INDEX_DIR.relative_to(PROJECT_ROOT)}")


## Experimental parameters

All architectures, candidates, and seeds are evaluated. NRMSE is calculated task by task as RMSE divided by the corresponding target standard deviation (`ddof=1`) in the fitting partition. Validation and test observations therefore never define the normalization scale used for model selection.

Architectures whose ensemble validation RMSE is within 1% of the minimum are retained as eligible. Seed stability, validation RMSE, and a fixed architecture priority resolve the final task-level selection without using test results.


In [ ]:
RESOLUTIONS = [4, 12, 20, 30, 60]
TARGETS = ["temperature", "relative_humidity"]
HORIZONS_MINUTES = [60, 120, 240, 480]
HISTORY_HOURS = 24
FEATURE_SET = "SHT_TIME"
SEQUENCE_FEATURES = [
    "temperature", "relative_humidity",
    "hour_sin", "hour_cos", "day_of_year_sin", "day_of_year_cos",
]
ARCHITECTURES = ["MLP", "LSTM", "GRU", "TCN"]
ARCHITECTURE_PRIORITY = {"MLP": 1, "GRU": 2, "LSTM": 3, "TCN": 4}
SEEDS = [2026, 2027, 2028]
BATCH_SIZE = 128
MAXIMUM_EPOCHS = 80
EARLY_STOPPING_PATIENCE = 10
LEARNING_RATE_PATIENCE = 5
VALIDATION_TOLERANCE = 0.01

ARCHITECTURE_GRID = {
    "MLP": [
        {"hidden_units": [128, 64], "dropout": 0.2, "learning_rate": 0.001},
        {"hidden_units": [256, 128], "dropout": 0.3, "learning_rate": 0.0005},
    ],
    "LSTM": [
        {"recurrent_units": 64, "dense_units": 32, "dropout": 0.2, "learning_rate": 0.001},
        {"recurrent_units": 96, "dense_units": 48, "dropout": 0.3, "learning_rate": 0.0005},
    ],
    "GRU": [
        {"recurrent_units": 64, "dense_units": 32, "dropout": 0.2, "learning_rate": 0.001},
        {"recurrent_units": 96, "dense_units": 48, "dropout": 0.3, "learning_rate": 0.0005},
    ],
    "TCN": [
        {"filters": 32, "kernel_size": 3, "dilations": [1, 2, 4, 8], "dropout": 0.1, "dense_units": 32, "learning_rate": 0.001},
        {"filters": 64, "kernel_size": 3, "dilations": [1, 2, 4, 8], "dropout": 0.2, "dense_units": 48, "learning_rate": 0.0005},
    ],
}


## Common forecast origins and sequence construction

The notebook reads the five resolution datasets and the effective forecast origins produced by notebooks `03` and `04`. Required columns, duplicated origins, index limits, and chronological partitions are checked before fitting. Every architecture receives identical origins, historical windows, targets, and partitions.

Causal forward filling is performed before window extraction. Any unresolved missing values are replaced later with medians computed from the fitting partition only.


In [ ]:
def load_inputs(resolution_minutes):
    data_path = RESOLUTION_DIR / f"greenhouse_{resolution_minutes}min.csv"
    index_path = INDEX_DIR / f"effective_indices_{resolution_minutes}min.csv"
    if not data_path.exists() or not index_path.exists():
        raise FileNotFoundError(
            f"Missing inputs for {resolution_minutes} min. Run notebooks 03 and 04 first."
        )
    data = pd.read_csv(data_path, parse_dates=["timestamp"])
    data = data.sort_values("timestamp").reset_index(drop=True)
    origins = pd.read_csv(index_path, parse_dates=["origin_timestamp"])
    origins = origins.sort_values("origin_index").reset_index(drop=True)
    return data, origins


def build_sequences(data, origins, resolution_minutes):
    prepared = data[SEQUENCE_FEATURES].copy().ffill()
    values = prepared.to_numpy(dtype=np.float32)
    history_steps = HISTORY_HOURS * 60 // resolution_minutes
    sequences = np.empty(
        (len(origins), history_steps, len(SEQUENCE_FEATURES)), dtype=np.float32
    )
    for row_number, row in enumerate(origins.itertuples(index=False)):
        start = int(row.history_start_index)
        end = int(row.history_end_index) + 1
        window = values[start:end]
        if window.shape != (history_steps, len(SEQUENCE_FEATURES)):
            raise ValueError(
                f"Unexpected window shape at {resolution_minutes} min origin "
                f"{row.origin_index}: {window.shape}"
            )
        sequences[row_number] = window

    output_columns, outputs = [], []
    for target in TARGETS:
        for horizon in HORIZONS_MINUTES:
            output_columns.append(f"{target}__h{horizon}")
            positions = origins[f"target_index_h{horizon}"].astype(int).to_numpy()
            outputs.append(data.iloc[positions][target].to_numpy(dtype=np.float32))
    targets = np.column_stack(outputs).astype(np.float32)
    return sequences, targets, output_columns


datasets = {}
effective_origins = {}
sequence_matrices = {}
target_matrices = {}
output_names = None
sample_rows = []

for resolution in RESOLUTIONS:
    data, origins = load_inputs(resolution)
    required_data_columns = {"timestamp", *SEQUENCE_FEATURES, *TARGETS}
    missing_data_columns = sorted(required_data_columns.difference(data.columns))
    if missing_data_columns:
        raise KeyError(f"Missing data columns for {resolution} min: {missing_data_columns}")

    required_index_columns = {
        "origin_index", "origin_timestamp", "history_start_index", "history_end_index", "split",
        *{f"target_index_h{horizon}" for horizon in HORIZONS_MINUTES},
    }
    missing_index_columns = sorted(required_index_columns.difference(origins.columns))
    if missing_index_columns:
        raise KeyError(
            f"Missing index columns for {resolution} min: {missing_index_columns}"
        )

    indices_within_dataset = bool(
        origins["origin_index"].between(0, len(data) - 1).all()
        and origins["history_start_index"].between(0, len(data) - 1).all()
        and origins["history_end_index"].between(0, len(data) - 1).all()
        and all(
            origins[f"target_index_h{horizon}"].between(0, len(data) - 1).all()
            for horizon in HORIZONS_MINUTES
        )
    )
    duplicate_origins = int(origins["origin_index"].duplicated().sum())
    duplicate_timestamps = int(origins["origin_timestamp"].duplicated().sum())
    split_order = [
        origins.loc[origins["split"].eq(split), "origin_timestamp"]
        for split in ["train", "validation", "test"]
    ]
    chronological_splits = bool(
        all(not values.empty for values in split_order)
        and split_order[0].max() < split_order[1].min()
        and split_order[1].max() < split_order[2].min()
    )

    if not indices_within_dataset:
        raise IndexError(f"Forecast indices exceed the {resolution}-minute dataset.")
    if duplicate_origins or duplicate_timestamps:
        raise ValueError(f"Duplicate forecast origins found for {resolution} minutes.")
    if not chronological_splits:
        raise ValueError(f"Non-chronological data partitions found for {resolution} minutes.")

    X, Y, names = build_sequences(data, origins, resolution)
    datasets[resolution] = data
    effective_origins[resolution] = origins
    sequence_matrices[resolution] = X
    target_matrices[resolution] = Y
    output_names = names

    split_counts = origins["split"].value_counts()
    sample_rows.append({
        "resolution": f"{resolution}min",
        "dataset_rows": len(data),
        "history_steps": X.shape[1],
        "sequence_features": X.shape[2],
        "outputs": Y.shape[1],
        "total_origins": len(origins),
        "train": int(split_counts.get("train", 0)),
        "validation": int(split_counts.get("validation", 0)),
        "test": int(split_counts.get("test", 0)),
    })

sample_summary = pd.DataFrame(sample_rows)
sample_summary.to_csv(RESULTS_DIR / "01_input_sample_summary.csv", index=False)
feature_map = pd.DataFrame({
    "resolution": [f"{resolution}min" for resolution in RESOLUTIONS],
    "feature_set": [FEATURE_SET] * len(RESOLUTIONS),
    "sequence_features": [";".join(SEQUENCE_FEATURES)] * len(RESOLUTIONS),
})
feature_map.to_csv(RESULTS_DIR / "02_feature_map.csv", index=False)
display(sample_summary)


## Training-only preprocessing

The input scaler is fitted across all time steps belonging to fitting samples. The output scaler is fitted across the eight simultaneous targets. The fitted scalers, training medians, and column order are saved with the final models.


In [ ]:
def split_mask(origins, split):
    return origins["split"].eq(split).to_numpy()


def fit_preprocessor(X, Y, fit_mask):
    fitting_values = X[fit_mask].reshape(-1, X.shape[-1])
    medians = np.nanmedian(fitting_values, axis=0)
    if np.isnan(medians).any():
        missing_features = np.asarray(SEQUENCE_FEATURES)[np.isnan(medians)].tolist()
        raise ValueError(f"No finite fitting values for features: {missing_features}")
    fitting_values = np.where(np.isnan(fitting_values), medians, fitting_values)

    input_scaler = StandardScaler().fit(fitting_values)
    output_scaler = StandardScaler().fit(Y[fit_mask])
    return {
        "input_medians": medians.astype(np.float32),
        "input_scaler": input_scaler,
        "output_scaler": output_scaler,
        "sequence_features": SEQUENCE_FEATURES,
        "output_names": output_names,
        "history_hours": HISTORY_HOURS,
    }


def transform_inputs(X, preprocessor):
    medians = preprocessor["input_medians"]
    imputed = np.where(np.isnan(X), medians[None, None, :], X)
    flat = imputed.reshape(-1, imputed.shape[-1])
    scaled = preprocessor["input_scaler"].transform(flat)
    return scaled.reshape(imputed.shape).astype(np.float32)


def transform_outputs(Y, preprocessor):
    return preprocessor["output_scaler"].transform(Y).astype(np.float32)


def inverse_outputs(Y_scaled, preprocessor):
    return preprocessor["output_scaler"].inverse_transform(Y_scaled).astype(np.float32)


training_preprocessors = {}
scaled_training_data = {}
for resolution in RESOLUTIONS:
    origins = effective_origins[resolution]
    train = split_mask(origins, "train")
    preprocessor = fit_preprocessor(
        sequence_matrices[resolution], target_matrices[resolution], train
    )
    training_preprocessors[resolution] = preprocessor
    scaled_training_data[resolution] = (
        transform_inputs(sequence_matrices[resolution], preprocessor),
        transform_outputs(target_matrices[resolution], preprocessor),
    )


## Neural architectures

The MLP flattens the sequence. LSTM and GRU preserve the recurrent structure. The TCN uses causal residual convolutional blocks with dilation rates 1, 2, 4, and 8. All networks optimize mean squared error with Adam and produce eight linear outputs.


In [ ]:
def reset_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)


def tcn_residual_block(x, filters, kernel_size, dilation, dropout):
    residual = x
    x = layers.Conv1D(
        filters, kernel_size, padding="causal", dilation_rate=dilation,
        activation="relu", kernel_initializer="he_normal",
    )(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Conv1D(
        filters, kernel_size, padding="causal", dilation_rate=dilation,
        activation=None, kernel_initializer="he_normal",
    )(x)
    if residual.shape[-1] != filters:
        residual = layers.Conv1D(filters, 1, padding="same")(residual)
    x = layers.Add()([x, residual])
    return layers.Activation("relu")(x)


def build_model(architecture, input_shape, output_size, parameters, seed):
    keras.backend.clear_session()
    reset_seed(seed)
    inputs = keras.Input(shape=input_shape, name="history_sequence")

    if architecture == "MLP":
        x = layers.Flatten()(inputs)
        for units in parameters["hidden_units"]:
            x = layers.Dense(units, activation="relu")(x)
            x = layers.Dropout(parameters["dropout"])(x)
    elif architecture == "LSTM":
        x = layers.LSTM(
            parameters["recurrent_units"], dropout=parameters["dropout"]
        )(inputs)
        x = layers.Dense(parameters["dense_units"], activation="relu")(x)
        x = layers.Dropout(parameters["dropout"])(x)
    elif architecture == "GRU":
        x = layers.GRU(
            parameters["recurrent_units"], dropout=parameters["dropout"]
        )(inputs)
        x = layers.Dense(parameters["dense_units"], activation="relu")(x)
        x = layers.Dropout(parameters["dropout"])(x)
    elif architecture == "TCN":
        x = inputs
        for dilation in parameters["dilations"]:
            x = tcn_residual_block(
                x, parameters["filters"], parameters["kernel_size"],
                dilation, parameters["dropout"],
            )
        x = layers.GlobalAveragePooling1D()(x)
        x = layers.Dense(parameters["dense_units"], activation="relu")(x)
        x = layers.Dropout(parameters["dropout"])(x)
    else:
        raise ValueError(f"Unknown architecture: {architecture}")

    outputs = layers.Dense(output_size, activation="linear", name="multihorizon_outputs")(x)
    model = keras.Model(inputs=inputs, outputs=outputs, name=architecture.lower())
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=parameters["learning_rate"]),
        loss="mse",
    )
    return model


def validation_callbacks():
    return [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=EARLY_STOPPING_PATIENCE,
            restore_best_weights=True, mode="min",
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=LEARNING_RATE_PATIENCE,
            min_lr=1e-6, mode="min",
        ),
    ]


class EpochProgress(keras.callbacks.Callback):
    # Compact per-epoch progress bar compatible with silent Keras training.

    def __init__(self, total_epochs, description, position=1):
        super().__init__()
        self.progress = tqdm(
            total=int(total_epochs),
            desc=description,
            unit="epoch",
            leave=False,
            position=position,
            dynamic_ncols=True,
        )

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        postfix = {}
        if "loss" in logs:
            postfix["loss"] = f"{float(logs['loss']):.4f}"
        if "val_loss" in logs:
            postfix["val_loss"] = f"{float(logs['val_loss']):.4f}"
        if postfix:
            self.progress.set_postfix(postfix, refresh=False)
        self.progress.update(1)

    def on_train_end(self, logs=None):
        self.progress.clear()
        self.progress.close()
        if hasattr(self.progress, "container"):
            self.progress.container.close()


In [ ]:
def task_metrics(y_true, y_pred, normalization_scale):
    rows = []
    for target_index, target in enumerate(TARGETS):
        for horizon_index, horizon in enumerate(HORIZONS_MINUTES):
            output_index = target_index * len(HORIZONS_MINUTES) + horizon_index
            observed = y_true[:, output_index]
            predicted = y_pred[:, output_index]
            rmse = float(np.sqrt(mean_squared_error(observed, predicted)))
            scale = float(normalization_scale[output_index])
            rows.append({
                "target": target,
                "horizon_minutes": horizon,
                "n": len(observed),
                "rmse": rmse,
                "nrmse": rmse / scale if scale > 0 else np.nan,
                "r2": float(r2_score(observed, predicted)),
                "mae": float(mean_absolute_error(observed, predicted)),
                "bias": float(np.mean(predicted - observed)),
            })
    return pd.DataFrame(rows)


def prediction_rows(
    origins, y_true, y_pred, resolution, architecture, candidate_id, split
):
    selected = origins.loc[origins["split"].eq(split)].reset_index(drop=True)
    rows = []
    for target_index, target in enumerate(TARGETS):
        for horizon_index, horizon in enumerate(HORIZONS_MINUTES):
            output_index = target_index * len(HORIZONS_MINUTES) + horizon_index
            rows.append(pd.DataFrame({
                "resolution": f"{resolution}min",
                "resolution_minutes": resolution,
                "model": architecture,
                "family": "deep_learning",
                "architecture": architecture,
                "feature_set": FEATURE_SET,
                "candidate_id": candidate_id,
                "split": split,
                "origin_index": selected["origin_index"],
                "origin_timestamp": selected["origin_timestamp"],
                "target": target,
                "horizon_minutes": horizon,
                "observed": y_true[:, output_index],
                "predicted": y_pred[:, output_index],
            }))
    return pd.concat(rows, ignore_index=True)


## Phase A — candidate evaluation across independent seeds

Each candidate is trained separately with every seed. Candidate ranking uses mean validation NRMSE, followed by its standard deviation across seeds and mean validation $R^2$. The best epoch is determined independently for every seed from validation loss.

The display uses one overall progress bar and one temporary bar for the model currently being trained. Completed model bars are removed automatically.


In [ ]:
seed_validation_metric_frames = []
seed_run_rows = []
history_rows = []
validation_prediction_cache = {}

phase_a_total_runs = (
    len(RESOLUTIONS)
    * len(SEEDS)
    * sum(len(ARCHITECTURE_GRID[architecture]) for architecture in ARCHITECTURES)
)
phase_a_progress = tqdm(
    total=phase_a_total_runs,
    desc="Phase A — candidate models",
    unit="model",
    position=0,
    dynamic_ncols=True,
)

for resolution in RESOLUTIONS:
    origins = effective_origins[resolution]
    train = split_mask(origins, "train")
    validation = split_mask(origins, "validation")
    X_scaled, Y_scaled = scaled_training_data[resolution]
    Y = target_matrices[resolution]
    preprocessor = training_preprocessors[resolution]
    normalization_scale = np.std(Y[train], axis=0, ddof=1)

    for architecture in ARCHITECTURES:
        for candidate_id, parameters in enumerate(ARCHITECTURE_GRID[architecture], start=1):
            for seed in SEEDS:
                run_label = (
                    f"{resolution} min | {architecture} | "
                    f"candidate {candidate_id} | seed {seed}"
                )
                model = build_model(
                    architecture, X_scaled.shape[1:], Y_scaled.shape[1], parameters, seed
                )
                start = time.perf_counter()
                history = model.fit(
                    X_scaled[train], Y_scaled[train],
                    validation_data=(X_scaled[validation], Y_scaled[validation]),
                    epochs=MAXIMUM_EPOCHS,
                    batch_size=BATCH_SIZE,
                    callbacks=[
                        *validation_callbacks(),
                        EpochProgress(
                            MAXIMUM_EPOCHS,
                            run_label,
                            position=1,
                        ),
                    ],
                    verbose=0,
                    shuffle=True,
                )
                fit_seconds = time.perf_counter() - start
                best_epoch = int(np.argmin(history.history["val_loss"]) + 1)
                phase_a_progress.set_postfix({
                    "resolution": f"{resolution} min",
                    "architecture": architecture,
                    "candidate": candidate_id,
                    "seed": seed,
                    "epochs": len(history.history["loss"]),
                    "last_min": f"{fit_seconds / 60:.1f}",
                }, refresh=False)
                phase_a_progress.update(1)

                start = time.perf_counter()
                prediction_scaled = model.predict(
                    X_scaled[validation], batch_size=BATCH_SIZE, verbose=0
                )
                inference_seconds = time.perf_counter() - start
                prediction = inverse_outputs(prediction_scaled, preprocessor)
                validation_prediction_cache[(
                    resolution, architecture, candidate_id, seed
                )] = prediction
                metrics = task_metrics(Y[validation], prediction, normalization_scale).assign(
                    resolution=f"{resolution}min",
                    resolution_minutes=resolution,
                    architecture=architecture,
                    feature_set=FEATURE_SET,
                    candidate_id=candidate_id,
                    seed=seed,
                    split="validation",
                    best_epoch=best_epoch,
                    fit_seconds=fit_seconds,
                    inference_seconds=inference_seconds,
                    parameters=json.dumps(parameters, sort_keys=True),
                )
                seed_validation_metric_frames.append(metrics)
                seed_run_rows.append({
                    "resolution": f"{resolution}min",
                    "resolution_minutes": resolution,
                    "architecture": architecture,
                    "candidate_id": candidate_id,
                    "seed": seed,
                    "parameters": json.dumps(parameters, sort_keys=True),
                    "best_epoch": best_epoch,
                    "fit_seconds": fit_seconds,
                    "inference_seconds": inference_seconds,
                    "mean_normalized_rmse": float(metrics["nrmse"].mean()),
                    "mean_rmse": float(metrics["rmse"].mean()),
                    "mean_r2": float(metrics["r2"].mean()),
                    "minimum_r2": float(metrics["r2"].min()),
                })
                for epoch, (loss, val_loss) in enumerate(
                    zip(history.history["loss"], history.history["val_loss"]), start=1
                ):
                    history_rows.append({
                        "resolution": f"{resolution}min",
                        "architecture": architecture,
                        "candidate_id": candidate_id,
                        "seed": seed,
                        "training_scope": "train_with_validation_monitoring",
                        "epoch": epoch,
                        "loss": loss,
                        "val_loss": val_loss,
                    })
                del model
                keras.backend.clear_session()

phase_a_progress.close()
seed_validation_metrics = pd.concat(seed_validation_metric_frames, ignore_index=True)
seed_run_summary = pd.DataFrame(seed_run_rows)
training_histories = pd.DataFrame(history_rows)
seed_validation_metrics.to_csv(RESULTS_DIR / "03_seed_validation_metrics.csv", index=False)
training_histories.to_csv(RESULTS_DIR / "07_training_histories.csv", index=False)

candidate_summary = (
    seed_run_summary.groupby(
        ["resolution", "resolution_minutes", "architecture", "candidate_id", "parameters"],
        as_index=False,
    )
    .agg(
        mean_normalized_rmse=("mean_normalized_rmse", "mean"),
        sd_normalized_rmse=("mean_normalized_rmse", "std"),
        mean_rmse=("mean_rmse", "mean"),
        sd_rmse=("mean_rmse", "std"),
        mean_r2=("mean_r2", "mean"),
        sd_r2=("mean_r2", "std"),
        minimum_r2=("minimum_r2", "min"),
        median_best_epoch=("best_epoch", "median"),
        mean_fit_seconds=("fit_seconds", "mean"),
        mean_inference_seconds=("inference_seconds", "mean"),
        independent_runs=("seed", "nunique"),
    )
)
candidate_summary[["sd_normalized_rmse", "sd_rmse", "sd_r2"]] = candidate_summary[
    ["sd_normalized_rmse", "sd_rmse", "sd_r2"]
].fillna(0.0)
candidate_summary.to_csv(RESULTS_DIR / "04_hyperparameter_tuning.csv", index=False)
display(candidate_summary)


## Validation-only candidate and task selection

The best hyperparameter candidate is first retained within each architecture and temporal resolution using mean validation NRMSE across the eight tasks and three seeds. Ensemble validation predictions are then generated for those 20 retained configurations.

The final architecture is selected independently for each of the 40 forecasting tasks. Architectures within 1% of the minimum ensemble validation RMSE are eligible; lower RMSE dispersion across seeds, lower ensemble validation RMSE and the fixed priority `MLP → GRU → LSTM → TCN` resolve the decision. Test results are not available to this rule.


In [ ]:
best_candidate_rows = []
for (resolution, architecture), subset in candidate_summary.groupby(
    ["resolution_minutes", "architecture"], sort=False
):
    winner = subset.sort_values(
        ["mean_normalized_rmse", "sd_normalized_rmse", "mean_r2"],
        ascending=[True, True, False],
    ).iloc[0]
    best_candidate_rows.append(winner.to_dict())

architecture_summary = pd.DataFrame(best_candidate_rows)
architecture_summary.to_csv(
    RESULTS_DIR / "05_architecture_validation_summary.csv", index=False
)

validation_ensemble_metric_frames = []
validation_prediction_frames = []
for selected_candidate in architecture_summary.itertuples(index=False):
    resolution = int(selected_candidate.resolution_minutes)
    architecture = selected_candidate.architecture
    candidate_id = int(selected_candidate.candidate_id)
    origins = effective_origins[resolution]
    train = split_mask(origins, "train")
    validation = split_mask(origins, "validation")
    Y = target_matrices[resolution]
    ensemble_prediction = np.mean([
        validation_prediction_cache[(resolution, architecture, candidate_id, seed)]
        for seed in SEEDS
    ], axis=0)
    metrics = task_metrics(
        Y[validation], ensemble_prediction, np.std(Y[train], axis=0, ddof=1)
    ).assign(
        resolution=f"{resolution}min",
        resolution_minutes=resolution,
        architecture=architecture,
        feature_set=FEATURE_SET,
        candidate_id=candidate_id,
        split="validation",
        epochs=max(1, int(round(selected_candidate.median_best_epoch))),
        independent_runs=len(SEEDS),
    )
    validation_ensemble_metric_frames.append(metrics)
    validation_prediction_frames.append(prediction_rows(
        origins, Y[validation], ensemble_prediction,
        resolution, architecture, candidate_id, "validation"
    ))

validation_ensemble_metrics = pd.concat(
    validation_ensemble_metric_frames, ignore_index=True
)
validation_predictions = pd.concat(validation_prediction_frames, ignore_index=True)

task_columns = ["resolution_minutes", "target", "horizon_minutes"]
seed_stability = (
    seed_validation_metrics.groupby(
        task_columns + ["architecture", "candidate_id"], as_index=False
    )
    .agg(
        validation_seed_rmse_sd=("rmse", "std"),
        independent_seeds=("seed", "nunique"),
    )
)
seed_stability["validation_seed_rmse_sd"] = (
    seed_stability["validation_seed_rmse_sd"].fillna(0.0)
)

task_candidates = validation_ensemble_metrics.merge(
    seed_stability,
    on=task_columns + ["architecture", "candidate_id"],
    how="left",
    validate="one_to_one",
)
task_candidates["minimum_validation_rmse"] = (
    task_candidates.groupby(task_columns)["rmse"].transform("min")
)
task_candidates["relative_gap_to_minimum"] = (
    task_candidates["rmse"] / task_candidates["minimum_validation_rmse"] - 1.0
)
task_candidates["within_one_percent"] = (
    task_candidates["relative_gap_to_minimum"] <= VALIDATION_TOLERANCE + 1e-12
)
task_candidates["architecture_priority"] = (
    task_candidates["architecture"].map(ARCHITECTURE_PRIORITY).fillna(999)
)

eligible = task_candidates.loc[task_candidates["within_one_percent"]].copy()
selected_task_configuration = (
    eligible.sort_values(
        task_columns + ["validation_seed_rmse_sd", "rmse", "architecture_priority"],
        ascending=[True, True, True, True, True, True],
    )
    .groupby(task_columns, as_index=False, sort=False)
    .first()
    .rename(columns={"rmse": "validation_rmse", "r2": "validation_r2"})
)
selected_task_configuration["selection_scope"] = "resolution × target × horizon"
selected_task_configuration["selection_basis"] = (
    "within 1% of minimum validation ensemble RMSE; then seed RMSE SD, "
    "validation RMSE and fixed architecture priority"
)
selected_task_configuration["test_used_for_selection"] = False
selected_task_configuration.to_csv(
    RESULTS_DIR / "06_selected_task_configuration.csv", index=False
)

display(architecture_summary.sort_values(["resolution_minutes", "mean_normalized_rmse"]))
display(selected_task_configuration[[
    "resolution", "target", "horizon_minutes", "architecture", "candidate_id",
    "validation_rmse", "validation_seed_rmse_sd", "relative_gap_to_minimum",
]])


## Final refitting and independent test evaluation

For each retained architecture–candidate pair, the fixed epoch count is the median best epoch across its three validation runs. Each seed is refitted on `train + validation`; test predictions are averaged across seeds before the primary metrics are calculated.

Only architecture–resolution pairs selected for at least one forecasting task are saved. A common preprocessing object is saved once per resolution. Complete validation and test predictions for all retained configurations remain available for transparent comparison.


In [ ]:
test_metric_frames = []
seed_test_metric_frames = []
prediction_frames = []
final_run_rows = []
final_history_rows = []
saved_model_rows = []
preprocessor_rows = []

selected_pairs = set(
    selected_task_configuration[["resolution_minutes", "architecture"]]
    .drop_duplicates().itertuples(index=False, name=None)
)
final_total_runs = len(architecture_summary) * len(SEEDS)
final_progress = tqdm(
    total=final_total_runs,
    desc="Final refit — train + validation",
    unit="model",
    position=0,
    dynamic_ncols=True,
)

for resolution in RESOLUTIONS:
    origins = effective_origins[resolution]
    train = split_mask(origins, "train")
    validation = split_mask(origins, "validation")
    test = split_mask(origins, "test")
    fit_scope = train | validation
    X = sequence_matrices[resolution]
    Y = target_matrices[resolution]

    final_preprocessor = fit_preprocessor(X, Y, fit_scope)
    X_scaled = transform_inputs(X, final_preprocessor)
    Y_scaled = transform_outputs(Y, final_preprocessor)
    normalization_scale = np.std(Y[fit_scope], axis=0, ddof=1)

    preprocessor_path = PREPROCESSOR_DIR / f"{resolution}min" / "preprocessing.joblib"
    preprocessor_path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(final_preprocessor, preprocessor_path)
    preprocessor_rows.append({
        "resolution": f"{resolution}min",
        "resolution_minutes": resolution,
        "feature_set": FEATURE_SET,
        "fit_scope": "train+validation",
        "preprocessor_path": str(preprocessor_path.relative_to(PROJECT_ROOT)),
    })

    resolution_configurations = architecture_summary.loc[
        architecture_summary["resolution_minutes"].eq(resolution)
    ]
    for selected_candidate in resolution_configurations.itertuples(index=False):
        architecture = selected_candidate.architecture
        candidate_id = int(selected_candidate.candidate_id)
        parameters = json.loads(selected_candidate.parameters)
        epochs = max(1, int(round(selected_candidate.median_best_epoch)))
        seed_predictions = []

        for seed in SEEDS:
            run_label = (
                f"{resolution} min | {architecture} | "
                f"candidate {candidate_id} | seed {seed}"
            )
            model = build_model(
                architecture, X_scaled.shape[1:], Y_scaled.shape[1], parameters, seed
            )
            start = time.perf_counter()
            history = model.fit(
                X_scaled[fit_scope], Y_scaled[fit_scope],
                epochs=epochs,
                batch_size=BATCH_SIZE,
                callbacks=[EpochProgress(epochs, run_label, position=1)],
                verbose=0,
                shuffle=True,
            )
            fit_seconds = time.perf_counter() - start
            final_progress.set_postfix({
                "resolution": f"{resolution} min",
                "architecture": architecture,
                "candidate": candidate_id,
                "seed": seed,
                "epochs": len(history.history["loss"]),
                "last_min": f"{fit_seconds / 60:.1f}",
            }, refresh=False)
            final_progress.update(1)

            start = time.perf_counter()
            prediction_scaled = model.predict(
                X_scaled[test], batch_size=BATCH_SIZE, verbose=0
            )
            inference_seconds = time.perf_counter() - start
            prediction = inverse_outputs(prediction_scaled, final_preprocessor)
            seed_predictions.append(prediction)

            seed_metrics = task_metrics(Y[test], prediction, normalization_scale).assign(
                resolution=f"{resolution}min",
                resolution_minutes=resolution,
                architecture=architecture,
                feature_set=FEATURE_SET,
                candidate_id=candidate_id,
                seed=seed,
                split="test",
                epochs=epochs,
                fit_seconds=fit_seconds,
                inference_seconds=inference_seconds,
            )
            seed_test_metric_frames.append(seed_metrics)
            final_run_rows.append({
                "resolution": f"{resolution}min",
                "resolution_minutes": resolution,
                "architecture": architecture,
                "candidate_id": candidate_id,
                "seed": seed,
                "epochs": epochs,
                "fit_seconds": fit_seconds,
                "inference_seconds": inference_seconds,
            })
            for epoch, loss in enumerate(history.history["loss"], start=1):
                final_history_rows.append({
                    "resolution": f"{resolution}min",
                    "architecture": architecture,
                    "candidate_id": candidate_id,
                    "seed": seed,
                    "training_scope": "train_plus_validation_fixed_epochs",
                    "epoch": epoch,
                    "loss": loss,
                    "val_loss": np.nan,
                })

            if (resolution, architecture) in selected_pairs:
                model_path = (
                    MODEL_DIR / f"{resolution}min" / architecture / FEATURE_SET
                    / f"seed_{seed}.keras"
                )
                model_path.parent.mkdir(parents=True, exist_ok=True)
                model.save(model_path)
                saved_model_rows.append({
                    "resolution": f"{resolution}min",
                    "resolution_minutes": resolution,
                    "architecture": architecture,
                    "feature_set": FEATURE_SET,
                    "candidate_id": candidate_id,
                    "seed": seed,
                    "epochs": epochs,
                    "model_path": str(model_path.relative_to(PROJECT_ROOT)),
                })
            del model
            keras.backend.clear_session()

        ensemble_prediction = np.mean(seed_predictions, axis=0)
        ensemble_metrics = task_metrics(
            Y[test], ensemble_prediction, normalization_scale
        ).assign(
            resolution=f"{resolution}min",
            resolution_minutes=resolution,
            architecture=architecture,
            feature_set=FEATURE_SET,
            candidate_id=candidate_id,
            split="test",
            epochs=epochs,
            independent_runs=len(SEEDS),
        )
        test_metric_frames.append(ensemble_metrics)
        prediction_frames.append(prediction_rows(
            origins, Y[test], ensemble_prediction,
            resolution, architecture, candidate_id, "test"
        ))

final_progress.close()
test_metrics = pd.concat(test_metric_frames, ignore_index=True)
seed_test_metrics = pd.concat(seed_test_metric_frames, ignore_index=True)
test_predictions = pd.concat(prediction_frames, ignore_index=True)
dl_metrics = pd.concat([validation_ensemble_metrics, test_metrics], ignore_index=True)
dl_predictions = pd.concat([validation_predictions, test_predictions], ignore_index=True)
final_runs = pd.DataFrame(final_run_rows)
training_histories = pd.concat(
    [training_histories, pd.DataFrame(final_history_rows)], ignore_index=True
)
saved_models = pd.DataFrame(saved_model_rows)
preprocessors = pd.DataFrame(preprocessor_rows)

selected_test_metrics = test_metrics.merge(
    selected_task_configuration[
        task_columns + ["architecture", "candidate_id"]
    ],
    on=task_columns + ["architecture", "candidate_id"],
    how="inner",
    validate="one_to_one",
)
selected_test_predictions = test_predictions.merge(
    selected_task_configuration[
        task_columns + ["architecture", "candidate_id"]
    ],
    on=task_columns + ["architecture", "candidate_id"],
    how="inner",
    validate="many_to_one",
)

dl_metrics.to_csv(RESULTS_DIR / "09_dl_metrics.csv", index=False)
dl_predictions.to_csv(RESULTS_DIR / "10_dl_predictions.csv", index=False)
seed_test_metrics.to_csv(RESULTS_DIR / "11_seed_test_metrics.csv", index=False)
saved_models.to_csv(RESULTS_DIR / "12_saved_model_catalog.csv", index=False)
preprocessors.to_csv(RESULTS_DIR / "13_preprocessor_catalog.csv", index=False)
selected_test_metrics.to_csv(RESULTS_DIR / "14_selected_dl_test_metrics.csv", index=False)
selected_test_predictions.to_csv(
    RESULTS_DIR / "15_selected_dl_test_predictions.csv", index=False
)
training_histories.to_csv(RESULTS_DIR / "07_training_histories.csv", index=False)


In [ ]:
refit_configuration = architecture_summary[[
    "resolution", "resolution_minutes", "architecture", "candidate_id", "parameters",
    "mean_normalized_rmse", "sd_normalized_rmse", "mean_rmse", "mean_r2",
    "median_best_epoch",
]].copy()
refit_configuration["feature_set"] = FEATURE_SET
refit_configuration["independent_seeds"] = ";".join(map(str, SEEDS))
refit_configuration["test_used_for_selection"] = False
refit_configuration.to_csv(
    RESULTS_DIR / "08_refit_configuration_by_resolution_architecture.csv", index=False
)

computational_summary = (
    final_runs.groupby(["architecture"], as_index=False)
    .agg(
        mean_fit_seconds=("fit_seconds", "mean"),
        total_fit_seconds=("fit_seconds", "sum"),
        mean_inference_seconds=("inference_seconds", "mean"),
        total_inference_seconds=("inference_seconds", "sum"),
        mean_epochs=("epochs", "mean"),
        independent_runs=("seed", "nunique"),
    )
)
computational_summary.to_csv(
    RESULTS_DIR / "16_computational_summary.csv", index=False
)

architecture_distribution = (
    selected_task_configuration["architecture"]
    .value_counts()
    .rename_axis("architecture")
    .reset_index(name="selected_tasks")
)
architecture_distribution.to_csv(
    RESULTS_DIR / "17_selected_architecture_distribution.csv", index=False
)

display(refit_configuration)
display(selected_task_configuration[[
    "resolution", "target", "horizon_minutes", "architecture", "candidate_id",
    "validation_rmse", "validation_seed_rmse_sd",
]])
display(architecture_distribution)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
sns.lineplot(
    data=architecture_summary,
    x="resolution_minutes", y="mean_normalized_rmse",
    hue="architecture", marker="o", ax=axes[0],
)
axes[0].set_title("Architecture comparison on validation data")
axes[0].set_xlabel("Temporal resolution (min)")
axes[0].set_ylabel("Mean validation NRMSE")

sns.lineplot(
    data=selected_test_metrics,
    x="horizon_minutes", y="rmse",
    hue="target", style="resolution", markers=True, dashes=False, ax=axes[1],
)
axes[1].set_title("Validation-selected deep-learning configurations")
axes[1].set_xlabel("Forecast horizon (min)")
axes[1].set_ylabel("Test RMSE")
axes[1].legend(fontsize=7, ncol=2)

summary_png = FIGURE_DIR / "06_deep_learning_summary.png"
summary_pdf = FIGURE_DIR / "06_deep_learning_summary.pdf"
fig.savefig(summary_png, dpi=300, bbox_inches="tight")
fig.savefig(summary_pdf, bbox_inches="tight")
plt.show()

selected_history_keys = (
    selected_task_configuration[
        ["resolution", "architecture", "candidate_id"]
    ].drop_duplicates()
)
selected_histories = training_histories.merge(
    selected_history_keys,
    on=["resolution", "architecture", "candidate_id"],
    how="inner",
)
selected_histories = selected_histories.loc[
    selected_histories["training_scope"].eq("train_with_validation_monitoring")
]
g = sns.relplot(
    data=selected_histories, x="epoch", y="val_loss",
    hue="architecture", style="seed", col="resolution", col_wrap=3,
    kind="line", facet_kws={"sharex": False, "sharey": False},
    height=3.2, aspect=1.2,
)
g.set_axis_labels("Epoch", "Validation loss")
g.set_titles("{col_name}")
training_png = FIGURE_DIR / "06_selected_training_curves.png"
training_pdf = FIGURE_DIR / "06_selected_training_curves.pdf"
g.figure.savefig(training_png, dpi=300, bbox_inches="tight")
g.figure.savefig(training_pdf, bbox_inches="tight")
plt.show()

print(f"Saved: {summary_png.relative_to(PROJECT_ROOT)}")
print(f"Saved: {summary_pdf.relative_to(PROJECT_ROOT)}")
print(f"Saved: {training_png.relative_to(PROJECT_ROOT)}")
print(f"Saved: {training_pdf.relative_to(PROJECT_ROOT)}")


In [ ]:
output_summary = pd.DataFrame({
    "artifact": [
        "retained architecture candidates",
        "validation-selected forecasting tasks",
        "selected test metric rows",
        "selected test prediction rows",
        "saved neural-network models",
        "saved preprocessing objects",
        "figure files",
    ],
    "count": [
        len(refit_configuration),
        len(selected_task_configuration),
        len(selected_test_metrics),
        len(selected_test_predictions),
        len(saved_models),
        len(preprocessors),
        sum(path.exists() for path in [summary_png, summary_pdf, training_png, training_pdf]),
    ],
})
output_summary.to_csv(RESULTS_DIR / "18_output_summary.csv", index=False)

display(output_summary)
print(f"Results: {RESULTS_DIR.relative_to(PROJECT_ROOT)}")
print(f"Models: {MODEL_DIR.relative_to(PROJECT_ROOT)}")
print(f"Preprocessors: {PREPROCESSOR_DIR.relative_to(PROJECT_ROOT)}")
print(f"Figures: {FIGURE_DIR.relative_to(PROJECT_ROOT)}")
